In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.metrics import f1_score





I successivi blocchi avranno l'obiettivo di ottenre le metriche citate nel documento di riferimento.
1. Raccolta del train set
2. Pipeline customizzata su train con logistic
3. Fine-tuning (opzionale)  --> ricerca dei migliori iperparametri con validation set
5. prova finale su test set

In [2]:
#aggiornata la lettura a partire da index 0, saltando la colonna index
train_dataset = pd.read_csv('../../assets/seeds_11/train_11.csv', index_col=0) 
val_dataset = pd.read_csv('../../assets/seeds_11/val_11.csv', index_col=0) 
test_dataset = pd.read_csv('../../assets/seeds_11/test_11.csv', index_col=0) 
train_dataset.head()
train_dataset.info()
train_dataset.describe()

<class 'pandas.DataFrame'>
RangeIndex: 87447 entries, 0 to 87446
Columns: 514 entries, Unnamed: 0 to label
dtypes: float64(512), int64(2)
memory usage: 342.9 MB


,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,label
count,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,...,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000
mean,54758.958020,0.679764,0.849554,0.445289,2.223606,0.402982,0.150932,0.787322,0.950320,1.445799,...,1.292553,0.069404,1.363546,0.854033,0.747607,0.793423,0.943036,1.012487,1.951402,1.678731
std,31526.549025,0.536190,0.469740,0.377183,0.651790,0.390675,0.247751,0.378588,0.581192,0.565572,...,0.647304,0.128133,0.751937,0.430971,0.547479,0.444029,0.749935,0.491253,0.822073,1.357813
min,0.000000,0.000000,0.091477,0.000000,0.000000,0.000000,0.000000,0.007958,0.000000,0.007696,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,27479.500000,0.249447,0.493474,0.141849,1.773298,0.123679,0.001104,0.515237,0.516021,1.033248,...,0.808194,0.000859,0.819211,0.535384,0.331774,0.467188,0.358122,0.653016,1.370313,0.000000
50%,54766.000000,0.569162,0.814180,0.364473,2.215324,0.297666,0.040499,0.760290,0.884864,1.406150,...,1.232570,0.020790,1.255924,0.788868,0.645804,0.717999,0.770445,0.968583,1.942727,2.000000
75%,82038.500000,0.996849,1.155845,0.659865,2.653792,0.559518,0.190120,1.021999,1.302694,1.811211,...,1.715715,0.079577,1.780249,1.104565,1.044532,1.041276,1.352082,1.318243,2.511427,3.000000
max,109308.000000,4.149904,3.961852,3.237006,5.986111,4.305758,2.643863,3.651450,4.827925,4.069973,...,4.895092,2.531628,5.641250,3.501384,4.882216,3.865492,5.966918,4.273132,5.906068,3.000000


Creazione di una pipeline che imposti:
1. Lo scaler prefissato
2. Numero componenti PCA
3. Fit su modello Logit Regressor

Restituisce il modello addestrato sul train set con le specifiche suddette

In [10]:
from utils.models import create_pretrained_pipeline
import skops.io as sio
from utils.models import load_all_skops
from utils import SRC_ROOT, ASSETS_DIR
from pathlib import Path



print(SRC_ROOT)
objs = load_all_skops(ASSETS_DIR / "seeds_11") 
scaler : StandardScaler = objs['scaler']
objs.pop('scaler')  # Rimuoviamo lo scaler dagli oggetti caricati, poiché lo abbiamo già separato


/home/kirjia/ProgettoQuantumBioetria/src
Caricamento di: scaler.skops...
scaler caricato con successo.
Caricamento di: pca_16.skops...
pca_16 caricato con successo.
Caricamento di: pca_4.skops...
pca_4 caricato con successo.
Caricamento di: pca_32.skops...
pca_32 caricato con successo.
Caricamento di: pca_8.skops...
pca_8 caricato con successo.


,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [5]:
# Prepara i dati per il training, la validazione e il test, separando le features dalle labels

X_train = train_dataset.drop(columns=['label'])
y_train = train_dataset['label']
X_val = val_dataset.drop(columns=['label']) 
y_val = val_dataset['label']
X_test = test_dataset.drop(columns=['label']) 
y_test = test_dataset['label']
X_train.describe()

Addestramento modello con diverse componenti PCA e calcolo metriche su validation set

F1_SCORE, 
Macro-AUROC (OvR, probabilità), 
Balanced Accuracy (argmax), 
ECE (top-label, 15 bin, L1)

In [12]:
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

metric_results_f1_score = {}
metric_results_auroc = {}
metric_results_bal_acc = {}


for k, v in objs.items():
    model = create_pretrained_pipeline(scaler, v, LogisticRegression())
    
    # 1. training
    model.fit(X_train, y_train)
    
    # 2. validation evaluation
    y_val_pred = model.predict(X_val)    
    f1_score_val = f1_score(y_val, y_val_pred, average='macro')
    metric_results_f1_score[k] = f1_score_val # Per ogni componente PCA

print(metric_results_f1_score)

{32: 0.774785288760379, 16: 0.7020043421796142, 8: 0.6115305174556116, 4: 0.480229235516361}


Addestramento modello con diverse componenti PCA e calcolo metriche su validation set

Macro-AUROC (OvR, probabilità)

In [ ]:
from sklearn.metrics import roc_auc_score


metric_results_auroc = {}

for k in [32, 16, 8, 4]:
    model = create_pretrained_pipeline(scaler, v, LogisticRegression())
    
    model.fit(X_train, y_train)
    
    y_val_proba = model.predict_proba(X_val)
    f1_score_val = f1_score(y_val, y_val_pred, average='macro')
    auroc = roc_auc_score(
        y_val,
        y_val_proba,
        multi_class='ovr',
        average='macro'
    )
    bal_acc = balanced_accuracy_score(y_val, y_val_pred)

    

In [14]:
print(metric_results_auroc)

{32: 0.958362965349365, 16: 0.9371212994726068, 8: 0.9052986704800352, 4: 0.8604727058609755}


Addestramento modello con diverse componenti PCA e calcolo metriche su validation set

Balanced Accuracy (argmax)

In [15]:
from sklearn.metrics import balanced_accuracy_score


metric_results_bal_acc = {}

for k in [32, 16, 8, 4]:
    model = create_model_pipeline(k)
    
    
    metric_results_bal_acc[k] = bal_acc
    

print(metric_results_bal_acc)

{32: 0.7551462655587202, 16: 0.6790077278158455, 8: 0.5998054203357875, 4: 0.5004645994716668}


Addestramento modello con diverse componenti PCA e calcolo metriche su validation set

ECE (top-label, 15 bin, L1)

In [ ]:
for k, v in objs.items():
    model = create_pretrained_pipeline(scaler, v, LogisticRegression())
    
    # 1. training
    model.fit(X_train, y_train)
    
    # 2. validation evaluation
    y_val_pred = model.predict(X_val)    
    f1_score_val = f1_score(y_val, y_val_pred, average='macro')
    metric_results_f1_score[k] = f1_score_val # Per ogni componente PCA


for k in [32, 16, 8, 4]:
    model = create_pretrained_pipeline(scaler, v, LogisticRegression())
    
    model.fit(X_train, y_train)
    
    y_val_proba = model.predict_proba(X_val)
    f1_score_val = f1_score(y_val, y_val_pred, average='macro')
    auroc = roc_auc_score(
        y_val,
        y_val_proba,
        multi_class='ovr',
        average='macro'
    )
    bal_acc = balanced_accuracy_score(y_val, y_val_pred)
